In [ ]:
## Apresentação 6 - Cálculo Numérico
# O GÊMEO DIGITAL
**Grupo 1:** Bruno Lopes de Almeida Zuffo, 
João Victor Bozola Bosi, 
Lucas Antero Albuquerque, 
Matheus Marchi Baron, 
Natalia Yumi Watanabe, 
Victor Hugo Albertino e Silva.

---

## 1. Introdução: Integração dos Três Subsistemas

Neste capítulo final, unificamos os três modelos físicos desenvolvidos ao longo do curso — a **Rede Hidráulica** (Cap. 1), a **Placa Térmica** (Cap. 2) e a **Membrana Elástica** (Cap. 3) — em um único **Gêmeo Digital (GD)**, seguindo a dependência sequencial estabelecida na Seção 6.1 do PDF:

1. **Placa Térmica**: resolve-se o balanço de calor no sólido, considerando que a proximidade dos microcanais modifica localmente a condutividade térmica efetiva (mesma formulação do Capítulo 4, Equação 4.3);
2. **Viscosidade / Condutância dos Canais**: a partir do campo de temperatura resultante, calcula-se a temperatura média de cada canal (integral de linha ao longo da aresta) e, com ela, a viscosidade local do fluido e a condutância hidráulica atualizada de cada microcanal;
3. **Rede Hidráulica**: monta-se a matriz de rigidez da rede com as condutâncias já corrigidas termicamente;
4. **Acoplamento Monolítico**: a rede resultante é acoplada, de forma monolítica (mesma estratégia do Capítulo 5), à membrana elástica que veda o reservatório de expansão.

Todas as especificações físicas nominais usadas abaixo seguem exatamente a Seção 6.2 do PDF (geometria da placa, condições de contorno térmicas, largura dos microcanais, pressão de entrada, propriedades da membrana, etc.). A célula a seguir monta a base completa do Gêmeo Digital — `functionsGD.py` orquestra os quatro passos acima dentro da função `MontaGemeoDigital`.

In [ ]:
import os
import sys
import numpy as np
import matplotlib
matplotlib.use("TkAgg")
import matplotlib.pyplot as plt

# Em um notebook Jupyter a variável especial __file__ nem sempre existe
# (ela só é definida quando o código roda como um script .py "de verdade").
# Por isso, usamos os.getcwd() como alternativa: ao abrir este notebook a
# partir da pasta GemeoDigital, o diretório de trabalho do kernel já
# corresponde a essa pasta.
try:
    diretorio_atual = os.path.dirname(os.path.abspath(__file__))
except NameError:
    diretorio_atual = os.getcwd()

diretorio_pai = os.path.dirname(diretorio_atual)

SUBPASTAS_PROJETO = [
    "RedeHidraulica",
    "PlacaTermica",
    "MembranaElastica",
    "AcoplamentoHidraulicoTermico",
    "AcoplamentoHidraulicoMecanico",
]

if diretorio_pai not in sys.path:
    sys.path.insert(0, diretorio_pai)

for _sub in SUBPASTAS_PROJETO:
    _caminho_sub = os.path.join(diretorio_pai, _sub)
    if os.path.isdir(_caminho_sub) and _caminho_sub not in sys.path:
        sys.path.insert(0, _caminho_sub)

# O notebook já está dentro da pasta GemeoDigital, junto com functionsGD.py
from functionsGD import (
    MontaGemeoDigital, RodaSimulacaoBase, PlotaEstadoBaseGD,
    RodaExercicio1_FalhasHidraulicas, PlotaExercicio1_FalhasHidraulicas,
    RodaExercicio2_AnaliseDinamica, PlotaExercicio2_AnaliseDinamica,
    SimulaEnergiaDissipada, Interpola_Potencia_Linear, 
    Interpola_Potencia_Cubica, RodaExercicio645_Sensibilidade,
    PlotaExercicio645_Sensibilidade, Newton_Raphson_Energia, Regressao_Polinomial_Global,
    Analise_Ruido_Estocastico, SimulaCaso645
)

# ==============================================================================
# DEFINIÇÃO DOS PARÂMETROS INICIAIS DO SISTEMA
# ==============================================================================

params = {}

# ---- Conversoes e nivel de complexidade da rede ----
params['mm_to_m'] = 0.001
params['complex_level'] = 3          # Nivel de complexidade topologica: 3
params['n_inlet'] = 0                # No de entrada (Inlet 0)
params['n_outlet'] = 5               # No de saida (Outlet 5, espinha central)

# ---- Placa Termica ----
params['Lx'] = 0.03                          # Lx = 3 cm, em metros
params['Ly'] = 0.015                         # Ly = 1.5 cm, em metros
params['k0'] = 0.25                          # Condutividade da matriz solida: W/(K.m)
params['fonte_calor'] = 5.0e5                # Termo de fonte de calor: W/m^3
params['TL'] = 10.0                          # Temperatura no lado esquerdo: °C
params['TR'] = 30.0                          # Temperatura no lado direito: °C
params['R_incl'] = 0.25e-2                   # Raio da inclusao circular: 0.25 cm em m
params['xincl'] = 0.02 + params['R_incl']    # Centro em x: (2 + R) cm, em m
params['yincl'] = 0.75e-2                    # Centro em y: 0.75 cm, em m
params['TC'] = 35.0                          # Temperatura fixada na inclusao: °C

# Malha de discretizacao da placa termica e raio de influencia dos canais
# (compativel com as malhas usadas no acoplamento Hidraulico-Termico do Cap. 4)
params['Nx_t'] = 121
params['Ny_t'] = 61
params['d_max'] = 0.0005                     # Raio de proximidade (Equacao 4.3)

# ---- Rede Hidraulica ----
params['H_k'] = 1000e-6                      # Largura do canal (secao quadrada): 1000 um
params['p_inlet_dim'] = 5000.0               # Pressao prescrita no Inlet: 5000 Pa

# ---- Membrana Elastica ----
params['R_fisico'] = 0.25e-2                 # Raio fisico: 0.25 cm em m
params['e_esp'] = 0.1e-3                     # Espessura estrutural: 0.1 mm em m
params['sigma'] = 200.0                      # Tensao mecanica residual: N/m
params['rho'] = 900.0                        # Densidade de massa: kg/m^3
params['beta_ad'] = 0.1                      # Coeficiente de atrito viscoso (adimensional)

# Parametros adimensionais (base) e malha da membrana
params['Lx_ad'] = 2.0
params['Ly_ad'] = 2.0
params['R_ad'] = 1.0
params['sigma_ad'] = 1.0
params['rho_ad'] = 1.0
params['e_ad'] = 1.0
params['Nx_m'] = 51                          # Malha da membrana: 51 x 51 nos
params['Ny_m'] = 51

# Discretizacao temporal de verificacao
params['dt'] = 0.025
t_max_verificacao = 12.0

# ==============================================================================
# 2. MONTAGEM COMPLETA DO GEMEO DIGITAL
# ==============================================================================

print("=" * 80)
print("MONTAGEM DA BASE DO GEMEO DIGITAL (Capitulo 6)")
print("=" * 80)

print("Construindo Placa Termica, Rede Hidraulica e Membrana Elastica...")
GD = MontaGemeoDigital(params)


---
## 2. Investigando o Sistema via Amostragem Monte Carlo (Seção 6.3.2)

Ao longo da vida útil do dispositivo, os microcanais podem sofrer obstrução parcial por deposição de partículas ou bolhas presas, reduzindo sua condutância hidráulica. Como não sabemos, a priori, quais canais falharão nem o quanto, tratamos essa obstrução como um evento **estocástico**: cada canal tem probabilidade independente $p_O$ de estar obstruído e, quando isso ocorre, sua condutância nominal é reduzida por um fator de severidade $f_{obs}$:
$$C_k \leftarrow C_k / f_{obs}$$

Essa hipótese está implementada em `RandomFail(C_original, pO, fObs, rng)`, usada nos dois exercícios desta seção via **Método de Monte Carlo**.

### 2.1 Exercício 1 — Análise Estacionária de Falhas Hidráulicas

Consideramos primeiro o subsistema **isolado** (Rede Hidráulica + Placa Térmica, sem a membrana), em regime permanente, com pressão de descarga nula no reservatório ($p_{outlet}=0$). Queremos estimar

$$Prob\big(q_{inlet} < q_{crítico}\big), \qquad q_{crítico} = 1{,}25\times10^{-5}$$

A vazão de entrada $q_{inlet}$ de cada realização é obtida por "reação" a partir da matriz de condutância **não restrita** $A$ (montada com as condutâncias já obstruídas daquela realização):
$$q_{inlet} = \sum_j A_{0j}\, p_j$$

Como o estimador de Monte Carlo de uma probabilidade é, na prática, uma proporção amostral, seu erro padrão teórico decai como $\sqrt{\hat p(1-\hat p)/N}$ — a mesma taxa assintótica $N^{-1/2}$ discutida na Seção 6.3.1 do PDF (Figura 6.2) para o exemplo clássico de estimação de $\pi$. Usamos exatamente esse critério (meia-largura do intervalo de confiança de 95% da proporção, comparada a uma margem-alvo) para definir, de forma objetiva e reprodutível, o tamanho amostral mínimo $N_{min}$ a partir do qual a estimativa pode ser considerada estatisticamente estabilizada (`DeterminaTamanhoAmostralEstabilizacao`).

Por fim, varremos $p_O \in [0{,}05,\, 0{,}65]$ e comparamos dois fatores de severidade, $f_{obs}=5$ e $f_{obs}=10$.

In [ ]:
# ========================================================================
# SECAO 6.3.2 - EXERCICIO 1
# Análise Estacionária de Falhas Hidráulicas
# ========================================================================

print("\n" + "=" * 80)
print("EXERCÍCIO 1 (Seção 6.3.2) - Análise Estacionária de Falhas Hidráulicas")
print("=" * 80)

# Limite crítico de vazão de entrada, conforme a "Pergunta de Projeto" do PDF
q_critico = 1.25e-5

N_convergencia = 8000
pO_convergencia = 0.6

# (c) Varredura no domínio de obstrução individual solicitado pelo PDF
pO_grid = np.arange(0.05, 0.65 + 1e-9, 0.05)
N_final = 3000   # tamanho amostral "convergido" usado em cada ponto da varredura

resultados_ex1 = RodaExercicio1_FalhasHidraulicas(
    GD,
    q_critico=q_critico,
    N_convergencia=N_convergencia,
    pO_convergencia=pO_convergencia,
    fObs_lista=(5, 10),
    pO_grid=pO_grid,
    N_final=N_final,
    p_outlet=0.0,
)

# ---- Salva os dois graficos do Exercicio 1 como imagem, na mesma pasta
#      deste script (diretorio_atual), alem de exibi-los na tela --------
caminho_fig_convergencia = os.path.join(diretorio_atual, "ex1_convergencia_monte_carlo.png")
caminho_fig_varredura = os.path.join(diretorio_atual, "ex1_prob_vs_pO.png")

PlotaExercicio1_FalhasHidraulicas(
    resultados_ex1,
    save_path_convergencia=caminho_fig_convergencia,
    save_path_varredura=caminho_fig_varredura,
)

print(f"\nFiguras do Exercicio 1 salvas em:")
print(f"  {caminho_fig_convergencia}")
print(f"  {caminho_fig_varredura}")

print(f"\nResumo da varredura (N = {resultados_ex1['N_final']} por ponto):")
for fObs in resultados_ex1['fObs_lista']:
    print(f"\n  f_obs = {fObs}")
    for pO, prob in zip(resultados_ex1['pO_grid'], resultados_ex1['prob_vs_pO'][fObs]):
        print(f"    p_O = {pO:.2f}  ->  Prob(q_inlet < q_crit) = {prob*100:.2f}%")

print("\n" + "=" * 80)
print("EXERCÍCIO 1 (Seção 6.3.2) CONCLUÍDO.")
print("=" * 80)


### 2.2 Exercício 2 — Análise Dinâmica do Gêmeo Digital Completo

Elevamos agora a análise para o **acoplamento forte multifísico completo**: a mesma hipótese de falha estocástica dos microcanais (mesma `RandomFail`, mesmo $p_O$ representativo do Exercício 1) é aplicada ao sistema transiente **membrana + rede + placa térmica** (malha $51\times51$ da membrana).

O indicador de projeto agora é energético:
$$E = \int_0^{4} \mathcal{P}(t)\, dt, \qquad \mathcal{P}(t) = \mathbf{p}(t)^{\top}\,\mathbb{A}\,\mathbf{p}(t)$$

isto é, a energia total dissipada no reservatório ao longo da janela de operação, obtida pela regra do trapézio sobre a potência hidráulica instantânea $\mathcal{P}(t)$. Estimamos $Prob(E < E_{crítico})$, com $E_{crítico}=7{,}0$, para dois passos de tempo distintos ($\delta t = 0{,}05$ e $\delta t = 0{,}1$), o que permite avaliar também o impacto da discretização temporal sobre a precisão estatística do estimador.

In [ ]:
# ==============================================================================
# SEÇÃO 6.3.2 - EXERCÍCIO 2
# Análise Dinâmica do Gêmeo Digital Completo
# ==============================================================================

print("\n" + "=" * 80)
print("EXERCÍCIO 2 (Seção 6.3.2) - Análise Dinâmica do Gêmeo Digital")
print("=" * 80)

E_critico = 7.0
N_dinamica = 2000  

resultados_ex2 = RodaExercicio2_AnaliseDinamica(
    GD,
    E_critico=E_critico,
    pO=pO_convergencia,          # mesmo cenário representativo do Exercício 1 (p_O = 0.6)
    fObs_lista=(5, 10),
    dt_lista=(0.05, 0.1),
    N=N_dinamica,
    t_max=4.0,
)

caminho_fig_ex2 = os.path.join(diretorio_atual, "ex2_analise_dinamica.png")
PlotaExercicio2_AnaliseDinamica(resultados_ex2, save_path=caminho_fig_ex2)

print(f"\nFigura do Exercício 2 salva em:\n  {caminho_fig_ex2}")

print(f"\nResumo do Exercício 2 (N = {resultados_ex2['N']} por combinação):")
for fObs in resultados_ex2['fObs_lista']:
    for dt in resultados_ex2['dt_lista']:
        prob = resultados_ex2['prob'][(fObs, dt)]
        print(f"  f_obs={fObs:<3d}  dt={dt:<5}  ->  "
              f"Prob(E < {E_critico}) = {prob * 100:.2f}%")

print("\n" + "=" * 80)
print("EXERCÍCIO 2 (Seção 6.3.2) CONCLUÍDO.")
print("=" * 80)


---
## 3. Investigando o Sistema via Aproximação de Dados (Seção 6.4.3)

Nesta seção, tratamos a curva de potência instantânea $\mathcal{P}(t)$ do Gêmeo Digital nominal (sem falhas) como um conjunto de dados discretos observados em $t \in [0, 4]$, com passo $\delta t = 0{,}05$, e investigamos diferentes estratégias de **aproximação de dados**: interpolação local (linear e cúbica), regressão polinomial global e sensibilidade a ruído estocástico na condição de contorno de pressão.

### 3.1 Exercícios 1 e 2 — Interpolação Local (Linear e Cúbica)

Construímos duas famílias de interpoladores locais sobre os mesmos dados discretos $\big(t_i, \mathcal{P}(t_i)\big)$, obtidos simulando a trajetória **nominal** (sem falhas) do GD completo:

- **Spline Linear** (`Interpola_Potencia_Linear`): conecta os pontos por retas — simples e robusta, mas com derivada descontínua nos nós;
- **Spline Cúbica** (`Interpola_Potencia_Cubica`): garante suavidade ($C^1$/$C^2$) nas interfaces entre os subintervalos.

Comparamos visualmente as duas curvas na malha completa e em um *zoom* na região de interface, para evidenciar a diferença de suavidade entre os dois métodos.

In [ ]:
# ========================================================================
# SEÇÃO 6.4.3 - EXERCÍCIO 1
# Interpolação Linear Local
# SEÇÃO 6.4.3 - EXERCÍCIO 2
# Interpolação Cúbica Local
# ========================================================================

print("\n" + "-"*85)
print(" EXERCÍCIOS 1 E 2 (Seção 6.4.3): Interpolação linear e cúbca da potência")
print("-" * 85)

# 1. Definindo o vetor de tempo conforme exigido: t em [0, 4] com dt = 0.05
dt = 0.05
t_max_interp = 4.0

print("-> Simulando a trajetória nominal (sem falhas) do GD completo "
      f"para obter P(t) em [0, {t_max_interp}] com δt = {dt}...")
_E_nominal, P_dados, t_dados = SimulaEnergiaDissipada(
    GD, GD['C'], dt, t_max_interp, params['p_inlet_dim']
)
print(f"   Energia dissipada nominal: E = {_E_nominal:.4f}")

# 3. Gerando os modelos de interpolação
print("-> Construindo Splines Lineares (Ex 1)...")
interpolador_linear = Interpola_Potencia_Linear(t_dados, P_dados)

print("-> Construindo Splines Cúbicos (Ex 2)...")
interpolador_cubico = Interpola_Potencia_Cubica(t_dados, P_dados)

# 4. Avaliando os modelos em uma malha temporal bem fina para ver a "suavidade"
t_fino = np.linspace(t_dados[0], t_dados[-1], 500)
P_linear = interpolador_linear(t_fino)
P_cubico = interpolador_cubico(t_fino)


# ========================================================================
# 5. Renderização Gráfica Comparativa
# ========================================================================

# ------------------------------------------
# GRÁFICO 1: VISÃO GERAL (COMPLETO)
# ------------------------------------------
fig1, ax1 = plt.subplots(figsize=(10, 6))

# Pontos reais observados (Nós da malha)
ax1.plot(t_dados, P_dados, 'o', color='black', markersize=4, label='Dados Discretos Observados ($\\delta t = 0.05$)')

# Curvas de interpolação
ax1.plot(t_fino, P_linear, '--', color='dodgerblue', alpha=0.8, linewidth=1.5, label='Spline Linear (Ex 1)')
ax1.plot(t_fino, P_cubico, '-', color='crimson', alpha=0.8, linewidth=1.5, label='Spline Cúbico (Ex 2)')

# Formatação visual
ax1.set_title("Aproximação de Dados da Potência: Splines Lineares vs Cúbicos", fontweight='bold')
ax1.set_xlabel("Tempo adimensional ($t$)")
ax1.set_ylabel("Potência Instantânea $\\mathcal{P}(t)$")
ax1.legend(fontsize=11)
ax1.grid(True, linestyle='--', alpha=0.6)

fig1.tight_layout()
caminho_salvar_interp = os.path.join(diretorio_atual, "interpolacao_potencia_ex1_ex2.png")
fig1.savefig(caminho_salvar_interp, dpi=300)
print(f"✓ Gráfico de interpolação salvo em: {caminho_salvar_interp}")


# ------------------------------------------
# GRÁFICO 2: ZOOM NAS INTERFACES
# ------------------------------------------
fig2, ax2 = plt.subplots(figsize=(10, 6))

# Repetimos o plot na nova figura
ax2.plot(t_dados, P_dados, 'o', color='black', markersize=4, label='Dados Discretos Observados ($\\delta t = 0.05$)')
ax2.plot(t_fino, P_linear, '--', color='dodgerblue', alpha=0.8, linewidth=1.5, label='Spline Linear (Ex 1)')
ax2.plot(t_fino, P_cubico, '-', color='crimson', alpha=0.8, linewidth=1.5, label='Spline Cúbico (Ex 2)')

t_zoom_ini = t_dados[0] + 0.25 * (t_dados[-1] - t_dados[0])
t_zoom_fim = t_dados[0] + 0.50 * (t_dados[-1] - t_dados[0])
mascara_zoom = (t_fino >= t_zoom_ini) & (t_fino <= t_zoom_fim)
P_zoom = np.concatenate([P_cubico[mascara_zoom], P_linear[mascara_zoom]])
margem = 0.1 * (P_zoom.max() - P_zoom.min() + 1e-12)


ax2.set_xlim(1.4, 1.6)
ax2.set_ylim(0.0, 0.5)




# Formatação visual do zoom
ax2.set_title(f"Zoom ({1.4} a {1.6}s) - Analisando Suavidade nas Interfaces", fontweight='bold')
ax2.set_xlabel("Tempo adimensional ($t$)")
ax2.set_ylabel("Potência Instantânea $\\mathcal{P}(t)$")
ax2.legend(fontsize=11)
ax2.grid(True, linestyle='--', alpha=0.6)

fig2.tight_layout()
caminho_salvar_zoom = os.path.join(diretorio_atual, "interpolacao_potencia_zoom.png")
fig2.savefig(caminho_salvar_zoom, dpi=300)
print(f"✓ Gráfico com zoom salvo em: {caminho_salvar_zoom}")


plt.show()

print("\n" + "=" * 80)
print("EXERCÍCIOS 1 e 2 (Seção 6.4.3) CONCLUÍDO.")
print("=" * 80)


### 3.2 Exercício 3 — Regressão Polinomial Global

Diferente da interpolação local, a regressão polinomial global (`Regressao_Polinomial_Global`, mínimos quadrados via `np.polyfit`) ajusta **um único** polinômio de grau $m$ a todos os dados simultaneamente. Testamos $m \in [3, 15]$ e medimos a qualidade de cada ajuste pelo erro $L_2$ (Eq. 6.14 do PDF):
$$e = \sqrt{\int_{t_0}^{t_f} \big[p(t) - \mathcal{P}(t)\big]^2\, dt}$$
calculado pela regra do trapézio. Espera-se que o erro caia com o aumento de $m$ — mas o objetivo também é observar sinais de *overfitting* em graus muito altos, quando o polinômio passa a ajustar ruído numérico em vez da forma real da curva.

In [ ]:
# ========================================================================
# # SEÇÃO 6.4.3 - EXERCÍCIO 3
# Regressão Polinomial Global
# ========================================================================

print("\n" + "-"*85)
print(" EXERCÍCIO 3 (6.4.3): Regressão Polinomial Global (grau m ∈ [3, 15])")
print("-" * 85)

graus_ex3 = list(range(3, 16))

print(f"-> Ajustando regressão polinomial global para graus m = {graus_ex3[0]} a {graus_ex3[-1]}...")
resultados_ex3 = Regressao_Polinomial_Global(t_dados, P_dados, graus=graus_ex3)

# ---- Tabela de erros L2 ----
print(f"\n{'Grau m':>8}  {'Erro L2 (Eq. 6.14)':>20}")
print("-" * 32)
for m in graus_ex3:
    print(f"{m:>8d}  {resultados_ex3['erros_L2'][m]:>20.6e}")

# ---- Figura 1: curvas de aproximação sobrepostas aos dados ----
t_fino_ex3 = np.linspace(t_dados[0], t_dados[-1], 600)

fig3, ax3 = plt.subplots(figsize=(12, 6))
ax3.plot(t_dados, P_dados, 'ko', markersize=3, label='Dados discretos ($\\delta t = 0.05$)', zorder=5)

cmap_ex3 = plt.cm.rainbow(np.linspace(0, 1, len(graus_ex3)))
for cor, m in zip(cmap_ex3, graus_ex3):
    coefs = resultados_ex3['coeficientes'][m]
    ax3.plot(t_fino_ex3, np.polyval(coefs, t_fino_ex3),
             color=cor, linewidth=1.2, alpha=0.85, label=f'm = {m}')

ax3.set_title("Regressão Polinomial Global — Graus m ∈ [3, 15]", fontweight='bold')
ax3.set_xlabel("Tempo adimensional ($t$)")
ax3.set_ylabel("Potência Instantânea $\\mathcal{P}(t)$")
ax3.legend(fontsize=8, ncol=3, loc='upper right')
ax3.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

caminho_fig_ex3 = os.path.join(diretorio_atual, "ex3_regressao_polinomial.png")
plt.savefig(caminho_fig_ex3, dpi=300)
print(f"\n✓ Figura das curvas de regressão salva em: {caminho_fig_ex3}")

# ---- Figura 2: erro L2 vs grau ----
fig3b, ax3b = plt.subplots(figsize=(8, 5))
erros_lista = [resultados_ex3['erros_L2'][m] for m in graus_ex3]
ax3b.semilogy(graus_ex3, erros_lista, 'o-', color='steelblue', linewidth=2, markersize=6)
ax3b.set_title("Erro L2 vs. Grau do Polinômio (Regressão Global)", fontweight='bold')
ax3b.set_xlabel("Grau $m$")
ax3b.set_ylabel("Erro $e$ (Eq. 6.14) — escala log")
ax3b.set_xticks(graus_ex3)
ax3b.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

caminho_fig_ex3b = os.path.join(diretorio_atual, "ex3_erro_L2_vs_grau.png")
plt.savefig(caminho_fig_ex3b, dpi=300)
print(f"✓ Figura do erro L2 salva em: {caminho_fig_ex3b}")

plt.show()


#### Análise dos Resultados — Exercício 3

Diferentemente da interpolação, que passa exatamente por todos os pontos amostrados, a regressão polinomial global busca um único polinômio de grau $m$ que minimize o erro quadrático total sobre o intervalo. Testamos graus de $m = 3$ até $m = 15$, quantificando a qualidade do ajuste pelo erro L2 contínuo (Eq. 6.14):

$$e = \sqrt{\int_{t_0}^{t_f} \big[p(t) - \mathcal{P}(t)\big]^2\, dt}$$

A partir do gráfico, observa-se o fenômeno de underfitting para graus baixos ($m = 3, 4, 5$) e, conforme aumentamos o grau, o ajuste melhora progressivamente.

---
### 3.3 Exercício 4 — Sensibilidade a Ruídos Estocásticos

Na prática, os sensores de pressão estão sujeitos a incertezas de medição. Este exercício investiga como um ruído estocástico na entrada do sistema afeta a qualidade da regressão. A pressão de entrada é perturbada a cada passo de tempo por uma variação aleatória uniforme:

$$p_{inlet}(t) = \frac{p_{inlet}}{p_{ref}}\Big[1 + U(-0{,}15,\, 0{,}15)\Big]$$

ou seja, uma flutuação de até $\pm 15\%$ em torno do valor nominal, sorteada independentemente a cada instante.

Nesse sentido, foi feito $N=100$ realizações do sinal ruidoso de $\mathcal{P}(t)$ e, para cada uma, repetimos a regressão polinomial global do Exercício 3. Comparamos o erro $L_2$ médio (sobre as realizações) com o erro obtido no caso limpo, para cada grau $m$.

In [ ]:
# ======================================================================
# # SEÇÃO 6.4.3 - EXERCÍCIO 4
# Análise de Sensibilidade a Ruídos Estocásticos
# ======================================================================
print("\n" + "-"*85)
print(" EXERCÍCIO 4 (6.4.3): Sensibilidade a Ruídos em p_inlet(t)")
print("-" * 85)

N_realizacoes_ex4 = 100   # número de trajetórias ruidosas
graus_ex4 = list(range(3, 16))

print(f"-> Simulando {N_realizacoes_ex4} trajetórias com p_inlet ruidosa "
      f"[U(-0.15, 0.15)] e ajustando regressão polinomial (m ∈ [{graus_ex4[0]}, {graus_ex4[-1]}])...")

resultados_ex4 = Analise_Ruido_Estocastico(
    GD,
    dt=dt,
    t_max=t_max_interp,
    p_inlet_dim=params['p_inlet_dim'],
    graus=graus_ex4,
    N_realizacoes=N_realizacoes_ex4,
    seed=42,
)

tempos_ex4 = resultados_ex4['tempos']

# ---- Tabela comparativa: erro médio com e sem ruído ----
print(f"\n{'Grau m':>8}  {'Erro L2 (sem ruído)':>22}  {'Erro L2 médio (com ruído)':>26}")
print("-" * 62)
for m in graus_ex4:
    e_limpo = resultados_ex3['erros_L2'][m]
    e_ruidoso = resultados_ex4['erros_L2_medio'][m]
    print(f"{m:>8d}  {e_limpo:>22.6e}  {e_ruidoso:>26.6e}")

# ---- Figura 1: trajetórias ruidosas + regressão de cada realização ----
t_fino_ex4 = np.linspace(tempos_ex4[0], tempos_ex4[-1], 600)

fig4, axes4 = plt.subplots(1, 2, figsize=(14, 5))

# Painel esquerdo — sinal ruidoso vs. dado limpo + média
ax_esq = axes4[0]
for k, P_r in enumerate(resultados_ex4['P_realizacoes']):
    ax_esq.plot(tempos_ex4, P_r, color='gray', linewidth=0.6, alpha=0.5,
                label='Realização ruidosa' if k == 0 else None)
ax_esq.plot(tempos_ex4, resultados_ex4['P_ruidoso_medio'],
            color='darkorange', linewidth=2.0, label='Média das realizações')
ax_esq.plot(t_dados, P_dados, 'k--', linewidth=1.5, label='Dados limpos (nominal)')
ax_esq.set_title("Sinal de Potência com Ruído Estocástico", fontweight='bold')
ax_esq.set_xlabel("Tempo adimensional ($t$)")
ax_esq.set_ylabel("Potência Instantânea $\\mathcal{P}(t)$")
ax_esq.legend(fontsize=9)
ax_esq.grid(True, linestyle='--', alpha=0.5)

# Painel direito — erro L2 médio (ruidoso) vs. grau, comparado ao limpo
ax_dir = axes4[1]
erros_limpos = [resultados_ex3['erros_L2'][m] for m in graus_ex4]
erros_ruidosos_med = [resultados_ex4['erros_L2_medio'][m] for m in graus_ex4]
ax_dir.semilogy(graus_ex4, erros_limpos, 's--', color='steelblue',
                linewidth=2, markersize=6, label='Sem ruído')
ax_dir.semilogy(graus_ex4, erros_ruidosos_med, 'o-', color='crimson',
                linewidth=2, markersize=6, label='Com ruído (média)')
ax_dir.set_title("Erro L2 vs. Grau: Sensibilidade ao Ruído", fontweight='bold')
ax_dir.set_xlabel("Grau $m$")
ax_dir.set_ylabel("Erro $e$ (Eq. 6.14) — escala log")
ax_dir.set_xticks(graus_ex4)
ax_dir.legend(fontsize=10)
ax_dir.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
caminho_fig_ex4 = os.path.join(diretorio_atual, "ex4_sensibilidade_ruido.png")
plt.savefig(caminho_fig_ex4, dpi=300)
print(f"\n✓ Figura do Exercício 4 salva em: {caminho_fig_ex4}")
plt.show()

print("\n" + "=" * 80)
print("EXERCÍCIO 4 (Seção 6.4.3) CONCLUÍDO.")
print("=" * 80)


### 4.1 Exercício 1 — Análise Numérica de Sensibilidade

Para cada parâmetro varrido ($T_C \in [0, 250]\,°C$ e $H \in [500, 1500]\,\mu m$, 6 pontos cada), calculamos as derivadas numéricas dos indicadores de saída por dois esquemas de diferenças finitas:

- **Forward** (1ª ordem): $\dfrac{dE}{dx} \approx \dfrac{E(x+\varepsilon) - E(x)}{\varepsilon}$
- **Centrada** (2ª ordem, mais precisa): $\dfrac{dE}{dx} \approx \dfrac{E(x+\varepsilon) - E(x-\varepsilon)}{2\varepsilon}$

Para a sensibilidade em relação a $H$, comparamos ainda essas diferenças finitas com um **método direto** (`SimulaCaso645(calcular_direto_H=True)`): como a condutância hidráulica de cada canal escala com $H^4$ (Cap. 1), a matriz de rigidez da rede adimensional satisfaz $A(H) = H^4 A_0 \Rightarrow dA/dH = (4/H)\,A(H)$. Essa derivada analítica alimenta um sistema linear de sensibilidade, resolvido junto ao próprio sistema físico a cada passo de tempo, fornecendo uma referência semi-analítica independente para validar as diferenças finitas.

In [ ]:
# ==============================================================================
# SEÇÃO 6.4.5 Investigando o comportamento do sistema via diferenciação numérica
# ==============================================================================

# ========================================================================
# SEÇÃO 6.4.5 - EXERCÍCIO 1
# Análise Numérica de Sensibilidade
# ========================================================================

print("\n" + "-"*85)
print(" EXERCÍCIO 1 (Seção 6.4.5): Análise Numérica de Sensibilidade")
print("-" * 85)

resultados_ex645 = RodaExercicio645_Sensibilidade(
    params,
    t_max=4.0,
    dt=0.05
)

PlotaExercicio645_Sensibilidade(
    resultados_ex645,
    save_dir=diretorio_atual
)

print("\nResumo numérico - Sensibilidade em relação a TC:")
for i, TC in enumerate(resultados_ex645["TC"]["valores"]):
    print(
        f"  TC={TC:8.2f} °C | "
        f"E={resultados_ex645['TC']['E'][i]:.6e} | "
        f"qin(tf)={resultados_ex645['TC']['qin_tf'][i]:.6e} | "
        f"V(tf)={resultados_ex645['TC']['V_tf'][i]:.6e}"
    )

print("\nResumo numérico - Sensibilidade em relação a H:")
for i, H in enumerate(resultados_ex645["H"]["valores"]):
    print(
        f"  H={H:8.2f} µm | "
        f"E={resultados_ex645['H']['E'][i]:.6e} | "
        f"dE/dH centered={resultados_ex645['H']['dE_centered'][i]:.6e} | "
        f"dE/dH direto={resultados_ex645['H']['dE_direct'][i]:.6e}"
    )

print("\n" + "=" * 80)
print("EXERCÍCIO 1 (Seção 6.4.5) CONCLUÍDO.")
print("=" * 80)


### 4.2 Exercício 2 — Localização de Raízes via Newton-Raphson

Invertendo a pergunta: qual largura de microcanal $H$ produz uma energia dissipada alvo $E_{alvo}=7{,}5$? Definimos
$$f(H) = E(H) - E_{alvo}$$
e buscamos sua raiz pelo **Método de Newton-Raphson**, com a derivada $f'(H)$ aproximada por diferença finita *forward* a cada iteração (o que exige uma simulação completa do Gêmeo Digital por avaliação, já que $E(H)$ não possui forma fechada).

In [ ]:
# ========================================================================
# SEÇÃO 6.4.5 - EXERCÍCIO 2
# Localização de Raízes via Newton-Raphson
# ========================================================================

print("\n" + "-"*85)
print(" EXERCÍCIO 2 (Seção 6.4.5): Localização de Raízes via Newton-Raphson")
print("-" * 85)

# Vamos buscar a largura H que resulta em uma energia adimensional de 7.5
H_inicial = 800.0 # micrometros (chute inicial razoável baseado no domínio)
H_otimo = Newton_Raphson_Energia(params, H_inicial, E_alvo=7.5)

print(f"\nLargura geométrica ótima encontrada: H = {H_otimo:.4f} µm")

print("\n" + "=" * 80)
print("EXERCÍCIO 2 (Seção 6.4.5) CONCLUÍDO.")
print("=" * 80)


---
## Extra: Verificação de Robustez do Newton-Raphson

Como verificação adicional — não solicitada explicitamente pelo enunciado, mas importante para validar a solução do Exercício 2 — repetimos a busca de Newton-Raphson a partir de **seis chutes iniciais diferentes**, espalhados por todo o domínio físico de $H$, e conferimos se todos convergem para o mesmo valor de $H_{ótimo}$.

Também plotamos a curva $E(H)$ no domínio completo $[500, 1500]\,\mu m$: isso revela se $f(H)=0$ possui mais de uma raiz e ajuda a entender a robustez (ou fragilidade) do método frente a diferentes pontos de partida — inclusive explicando eventuais divergências entre os chutes iniciais testados.

In [ ]:
# TESTES PARA VALIDAR O EXERCÍCIO 2 (Newton-Raphson) ---------------------------------------------------


# TESTE DE ROBUSTEZ: Newton-Raphson a partir de diferentes chutes iniciais

print("\n" + "-"*85)
print(" TESTE DE ROBUSTEZ: Newton-Raphson a partir de diferentes chutes iniciais")
print("-" * 85)

chutes_iniciais = [550.0, 700.0, 800.0, 1000.0, 1200.0, 1450.0]
resultados_robustez = []

for H0 in chutes_iniciais:
    print(f"\n>>> Chute inicial: H0 = {H0:.1f} um")
    H_final = Newton_Raphson_Energia(params, H0, E_alvo=7.5)
    resultados_robustez.append((H0, H_final))

print("\n" + "=" * 50)
print(f"{'H_inicial [um]':>16}  |  {'H_otimo [um]':>14}")
print("-" * 50)
for H0, Hf in resultados_robustez:
    print(f"{H0:>16.1f}  |  {Hf:>14.4f}")
print("=" * 50)

# ---- Verificacao automatica de consistencia -----------------------------
valores_H_otimo = np.array([Hf for _, Hf in resultados_robustez])
dispersao = valores_H_otimo.max() - valores_H_otimo.min()
media_H_otimo = valores_H_otimo.mean()

print(f"\nH_otimo medio entre os {len(chutes_iniciais)} chutes: {media_H_otimo:.4f} um")
print(f"Dispersao maxima entre os resultados: {dispersao:.4e} um")

if dispersao < 1e-2:
    print("✓ ROBUSTO: todos os chutes convergiram para o mesmo H_otimo "
          "(dentro de 0.01 um). E(H) e bem comportada no dominio testado.")
else:
    print("⚠ ATENCAO: os resultados divergiram entre si. Isso pode indicar "
          "multiplas raizes, regiao nao-monotonica de E(H), ou um chute "
          "inicial fora da bacia de atracao do metodo de Newton-Raphson.")

print("\n" + "=" * 80)
print("TESTE DE ROBUSTEZ CONCLUÍDO.")
print("=" * 80)

# Investigação do teste de robustez

print("\n" + "-"*85)
print(" INVESTIGACAO: variacao de E(H) no dominio completo [500, 1500] um")
print("-" * 85)

H_fino = np.linspace(500.0, 1500.0, 41)   # resolucao de 25 um
E_fino = []

for H_val in H_fino:
    res = SimulaCaso645(params, H_um=H_val)
    E_fino.append(res["E"])
    print(f"   H = {H_val:8.2f} um  ->  E = {res['E']:.6f}")

E_fino = np.array(E_fino)

fig_inv, ax_inv = plt.subplots(figsize=(10, 6))
ax_inv.plot(H_fino, E_fino, 'o-', color='steelblue', markersize=4, linewidth=1.5,
            label='E(H) — varredura direta')
ax_inv.axhline(7.5, color='crimson', linestyle='--', linewidth=1.5,
               label='E_alvo = 7.5')

# Marca as duas raizes encontradas pelo Newton-Raphson
for H_raiz, cor, nome in [(630.57, 'darkorange', 'Raiz 1'),
                           (702.75, 'green', 'Raiz 2')]:
    ax_inv.axvline(H_raiz, color=cor, linestyle=':', linewidth=1.3, alpha=0.8,
                   label=fr'{nome}: H $\approx$ {H_raiz:.1f} µm')

ax_inv.set_xlabel('Largura geométrica do microcanal, H [µm]')
ax_inv.set_ylabel('Energia dissipada adimensional, E')
ax_inv.set_title('Curva E(H) — Investigação de raízes múltiplas de E(H) = 7.5')
ax_inv.grid(True, linestyle=':', alpha=0.6)
ax_inv.legend()
plt.tight_layout()

caminho_fig_investigacao = os.path.join(diretorio_atual, "investigacao_E_vs_H.png")
plt.savefig(caminho_fig_investigacao, dpi=200, bbox_inches='tight')
print(f"\n✓ Figura salva em: {caminho_fig_investigacao}")
plt.show()
